# Chapter 4 — Analytic Forward Sensitivity Analysis
**Companion text:** L. M. Arriola and J. M. Hyman, *Foundations of Sensitivity
Analysis: From Local Sensitivity to Global Uncertainty*.
Manuscript in preparation for submission to SIAM (2026).
Not submitted, not under review, not accepted for publication.

**Purpose:** Derive and compute the Forward Sensitivity Equations (FSE) for the SIR model. Verify FSE matches finite differences. Illustrate the proliferation problem.

**Key claims tested:**
- FSE matches FD to within numerical tolerance; time-varying SI shows beta dominates early epidemic; proliferation cost scales as m*(n+1) solves

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import warnings; warnings.filterwarnings('ignore')
FULL = False
print('Chapter 4: Analytic Forward Sensitivity Analysis')

In [ ]:
# ── SIR Jacobian and forcing terms ────────────────────────────────────────────
def sir_jac(S, I, c, beta, tau_R, N):
    """Model Jacobian dF/du for SIR (3x3 matrix)."""
    return np.array([
        [-c*beta*I/N,  -c*beta*S/N,   0],
        [ c*beta*I/N,   c*beta*S/N - 1/tau_R, 0],
        [0,            1/tau_R,          0]
    ])

def sir_forcing(S, I, c, beta, tau_R, N, param):
    """Forcing dF/dp_j for each parameter."""
    if param == 'c':    return np.array([-beta*S*I/N,  beta*S*I/N, 0])
    if param == 'beta': return np.array([-c*S*I/N,     c*S*I/N,   0])
    if param == 'tau_R':  return np.array([0, I/tau_R**2, -I/tau_R**2])
    return np.zeros(3)

def augmented_rhs(t, y_aug, c, beta, tau_R, N, param):
    """Augmented ODE: [dSIR/dt, dw/dt] where w = d(SIR)/d(param)."""
    S, I, R = y_aug[:3]
    w       = y_aug[3:]
    J       = sir_jac(S, I, c, beta, tau_R, N)
    Fp      = sir_forcing(S, I, c, beta, tau_R, N, param)
    dSIR = [-c*beta*S*I/N, c*beta*S*I/N - I/tau_R, I/tau_R]
    dw   = J @ w + Fp
    return np.concatenate([dSIR, dw])
print('FSE functions loaded.')

In [ ]:
# ── Verification: FSE matches finite differences ──────────────────────────────
c, beta, tau_R, N = 5.0, 0.06, 7.0, 1000
y0 = [999.0, 1.0, 0.0]

# FSE for beta
def run_fse(param):
    sol = solve_ivp(augmented_rhs, [0,30], y0+[0,0,0],
                    args=(c,beta,tau_R,N,param),
                    dense_output=True, rtol=1e-10, atol=1e-12)
    return sol.sol(30)

w_beta = run_fse('beta')[3:]

# Finite difference for beta
dp = 1e-6*beta
def sir_solve(b):
    sol = solve_ivp(lambda t,y: [-b*c*y[0]*y[1]/N, b*c*y[0]*y[1]/N-y[1]/tau_R, y[1]/tau_R],
                    [0,30], y0, dense_output=True, rtol=1e-10, atol=1e-12)
    return sol.sol(30)[:3]
fd_beta = (sir_solve(beta+dp) - sir_solve(beta-dp)) / (2*dp)

err = np.max(np.abs(w_beta - fd_beta))
print(f'FSE vs FD max error: {err:.2e}  (should be < 5e-6)')
assert err < 5e-6, f'FAIL: err={err:.2e}'  # FD truncation reaches ~2.4e-6; 5e-6 gives safety margin
print('PASS: FSE matches finite differences')

# SI_beta(I, t=30) via FSE
q0 = sir_solve(beta)[1]   # I(30)
SI_beta_fse = (beta / q0) * w_beta[1]
print(f'SI_beta(I, t=30) via FSE = {SI_beta_fse:.4f}')

In [ ]:
# ── Figure: Time-varying SI for SIR (Fig 4.2) ────────────────────────────────
t_vals = np.linspace(1, 90, 200)
sol_fse = solve_ivp(augmented_rhs, [0,90], y0+[0,0,0],
                    args=(c,beta,tau_R,N,'beta'),
                    dense_output=True, rtol=1e-10, atol=1e-12)
sol_fwd = solve_ivp(lambda t,y: [-c*beta*y[0]*y[1]/N, c*beta*y[0]*y[1]/N-y[1]/tau_R, y[1]/tau_R],
                    [0,90], y0, dense_output=True, rtol=1e-10, atol=1e-12)

I_t  = sol_fwd.sol(t_vals)[1]
w_I  = sol_fse.sol(t_vals)[4]   # dI/d(beta)
SI_t = (beta / np.maximum(I_t, 0.01)) * w_I

fig, ax = plt.subplots(figsize=(7,4))
ax.plot(t_vals, SI_t, 'steelblue', lw=2)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Time (days)', fontsize=12)
ax.set_ylabel(r'$\mathcal{S}_\beta^{I(t)}(t)$', fontsize=12)
ax.set_title(r'Time-varying SI: $\mathcal{S}_\beta^{I(t)}$ for SIR model (Fig 4.2)', fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('ch04_fig_timevarying_SI.pdf', dpi=150, bbox_inches='tight')
plt.show()
print('Fig 4.2 saved.')

In [ ]:
try:
    from google.colab import files
    files.download('ch04_fig_timevarying_SI.pdf')
except ImportError:
    print('Not in Colab — figure saved locally.')